# Chrome Enterprise Policy와 Custom Root CA를 사용하는 AgentCore Browser

이 Notebook에서는 [Amazon Bedrock AgentCore Browser](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-tool.html)의 새로운 기능 두 가지를 살펴봅니다.

- **Chrome enterprise policies**: Browser Agent를 승인된 domain으로 제한하고 위험한 브라우저 기능 비활성화
- **Custom root CA certificates**: Agent가 비공개 Certificate Authority를 사용하는 서비스에 연결할 수 있도록 지원

**Part 1**에서는 브라우저를 AWS 문서로 제한하는 Chrome policy를 생성하고, Playwright로 제한 사항을 확인한 후(허용된 URL은 성공하고 차단된 URL은 Chrome에서 거부) session recording으로 적용 결과를 검토합니다.

**Part 2**에서는 의도적으로 신뢰할 수 없는 certificate를 사용하는 공개 사이트 [badssl.com](https://badssl.com)을 통해 custom root CA certificate를 살펴보고, Code Interpreter 세션이 비공개 CA를 신뢰하도록 구성하는 방법을 보여 줍니다.

### 사전 요구 사항

이 Notebook을 실행하기 전에 다음 항목을 준비하세요.

1. **Python 3.10 이상** 설치
2. 다음 환경 변수가 설정된 **AWS 자격 증명 구성**:
   - `AWS_ACCESS_KEY_ID`
   - `AWS_SECRET_ACCESS_KEY`
   - `AWS_SESSION_TOKEN`(IAM Identity Center 또는 STS의 임시 자격 증명을 사용할 때 필요)
   - `AWS_REGION`(예: `us-west-2`)
3. Amazon Bedrock AgentCore Browser, Code Interpreter, Amazon S3, AWS Secrets Manager 및 IAM role 생성에 대한 **IAM 권한**. 전체 policy는 [README](README.md)를 참고하세요.
4. Anthropic Claude에 대해 활성화된 **Amazon Bedrock 모델 액세스**(선택 사항인 Strands Agent 셀에만 필요)

다음 명령을 실행해 자격 증명 구성을 확인할 수 있습니다.
```bash
aws sts get-caller-identity
```

> **중요:** AWS IAM Identity Center 또는 AWS STS의 임시 자격 증명을 사용하세요. 장기 access key를 사용하지 마세요.


### Dependency 설치

In [ ]:
!pip install -qU -r requirements.txt

### 구성

이 Notebook은 AWS account ID와 리전에서 S3 bucket 이름 및 IAM role ARN을 자동으로 생성합니다. Placeholder 값을 교체할 필요가 **없습니다**.

In [ ]:
import boto3
import json
import time
import asyncio
from botocore.exceptions import ClientError

session = boto3.Session()
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
REGION = session.region_name

# 파생된 이름으로 수동 교체 불필요
BUCKET_NAME = f"ac-browser-policy-demo-{ACCOUNT_ID}-{REGION}"
AC_ROLE_NAME = "ac-browser-policy-execution-role"
BROWSER_NAME = "docs_research_browser"
POLICY_KEY = "browser-policies/docs-only-policy.json"

# Client 생성
iam_client = boto3.client("iam")
s3_client = boto3.client("s3", region_name=REGION)
sm_client = boto3.client("secretsmanager", region_name=REGION)

print(f"Account: {ACCOUNT_ID}")
print(f"Region:  {REGION}")
print(f"Bucket:  {BUCKET_NAME}")
print(f"Role:    {AC_ROLE_NAME}")

### S3 bucket 생성

Chrome policy JSON file과 session recording을 저장할 S3 bucket이 없다면 생성합니다.

In [ ]:
try:
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket {BUCKET_NAME} already exists")
except ClientError:
    create_params = {"Bucket": BUCKET_NAME}
    if REGION != "us-east-1":
        create_params["CreateBucketConfiguration"] = {"LocationConstraint": REGION}
    s3_client.create_bucket(**create_params)
    print(f"Bucket {BUCKET_NAME} created in {REGION}")

### IAM execution role 생성

브라우저 세션과 Code Interpreter 세션을 실행할 때 Amazon Bedrock AgentCore가 수임하는 IAM role을 생성합니다. Role에는 다음 항목이 필요합니다.
- `bedrock-agentcore.amazonaws.com`의 수임을 허용하는 trust policy
- Policy file 및 session recording bucket에 대한 S3 권한

In [ ]:
# Trust policy: bedrock-agentcore.amazonaws.com이 service principal
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

# Policy file 및 session recording용 S3 권한
s3_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:PutObject",
                "s3:GetObject",
                "s3:GetObjectVersion",
                "s3:ListBucket",
                "s3:ListMultipartUploadParts",
                "s3:AbortMultipartUpload",
            ],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*",
            ],
        }
    ],
}

try:
    role_response = iam_client.create_role(
        RoleName=AC_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for AgentCore Browser chrome policy demo",
    )
    EXECUTION_ROLE_ARN = role_response["Role"]["Arn"]
    print(f"Created role: {EXECUTION_ROLE_ARN}")

    # S3 inline policy 연결
    iam_client.put_role_policy(
        RoleName=AC_ROLE_NAME,
        PolicyName="ac_browser_s3_policy",
        PolicyDocument=json.dumps(s3_policy),
    )
    print("Attached S3 policy")

except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        EXECUTION_ROLE_ARN = iam_client.get_role(RoleName=AC_ROLE_NAME)["Role"]["Arn"]
        print(f"Role already exists: {EXECUTION_ROLE_ARN}")
    else:
        raise

print(f"\nExecution Role ARN: {EXECUTION_ROLE_ARN}")

IAM role이 전파될 때까지 기다립니다.

In [ ]:
print("Waiting 10 seconds for IAM role propagation...")
time.sleep(10)
print("Done.")

---
## Part 1: Chrome Enterprise Policy

이 섹션에서는 다음 작업을 수행합니다.
1. 탐색 범위를 AWS 문서로 제한하는 Chrome enterprise policy 정의
2. Policy JSON을 Amazon S3에 업로드
3. Policy가 **managed** policy로 적용되는 사용자 지정 AgentCore Browser 생성
4. Playwright로 허용된 URL(성공)과 차단된 URL(Chrome에서 거부) 탐색

### 1단계: Chrome enterprise policy 생성 및 업로드

이 policy는 기본적으로 모든 URL을 차단하고 AWS 문서만 허용합니다. 또한 password manager, download, DevTools 및 autofill을 비활성화합니다.

사용 가능한 Chrome policy 전체 목록은 [Chrome Enterprise policy list](https://chromeenterprise.google/policies/)를 참고하세요.

> **중요 - CDP 호환성:** `DeveloperToolsAvailability`를 `2`(disabled)로 설정하지 마세요. 모든 AgentCore Browser 자동화는 Playwright의 `connect_over_cdp`를 통해 Chrome DevTools Protocol(CDP)을 사용합니다. 이 policy를 `2`로 설정하면 Chrome 수준에서 CDP가 비활성화되어 모든 자동화가 조용히 중단됩니다. Proxy 계층에서는 WebSocket 연결이 성공하지만 Chrome이 CDP 명령을 거부해 timeout이 발생합니다. 대신 `0`(allowed) 또는 `1`(allowed only for extensions)을 사용하세요.


In [ ]:
policy = {
    "URLBlocklist": ["*"],
    "URLAllowlist": [
        "docs.aws.amazon.com",
        ".aws.amazon.com",
        ".amazonaws.com",
    ],
    "PasswordManagerEnabled": False,
    "DownloadRestrictions": 3,
    "DeveloperToolsAvailability": 0,
    "BookmarkBarEnabled": False,
    "AutofillAddressEnabled": False,
    "AutofillCreditCardEnabled": False,
}

s3_client.put_object(
    Bucket=BUCKET_NAME,
    Key=POLICY_KEY,
    Body=json.dumps(policy, indent=2),
    ContentType="application/json",
)

print(f"Policy uploaded to s3://{BUCKET_NAME}/{POLICY_KEY}")
print("\nPolicy contents:")
print(json.dumps(policy, indent=2))

### 2단계: Managed policy와 session recording을 사용하는 브라우저 생성

모든 세션에 Chrome policy를 적용하는 사용자 지정 브라우저를 생성합니다. `enterprise_policies` parameter는 각각 `location`(JSON file의 S3 path)과 `type`을 포함한 policy object 목록을 받습니다.

- **`MANAGED`** - 브라우저 수준에서 적용되며 override할 수 없음(Chrome의 `/etc/chromium/policies/managed/`에 mapping)
- **`RECOMMENDED`** - 세션 수준에서 preference로 적용(Chrome의 `/etc/chromium/policies/recommended/`에 mapping)

나중에 AgentCore Console에서 세션을 재생할 수 있도록 session recording을 활성화합니다.

In [ ]:
from bedrock_agentcore.tools import BrowserClient

client = BrowserClient(REGION)

# 이전 실행 등으로 브라우저가 이미 있으면 재사용
# Policy를 update하려면 먼저 Cleanup 섹션을 실행한 다음
# 이 셀을 다시 실행해 새 policy가 적용된 브라우저 생성
try:
    print("Creating browser with managed policies and session recording...")
    response = client.create_browser(
        name=BROWSER_NAME,
        execution_role_arn=EXECUTION_ROLE_ARN,
        network_configuration={"networkMode": "PUBLIC"},
        enterprise_policies=[
            {
                "location": {
                    "s3": {
                        "bucket": BUCKET_NAME,
                        "prefix": POLICY_KEY,
                    }
                },
                "type": "MANAGED",
            }
        ],
        recording={
            "enabled": True,
            "s3Location": {
                "bucket": BUCKET_NAME,
                "prefix": "policy-demo",
            },
        },
        description="Browser restricted to AWS docs with Chrome enterprise policies",
    )
    browser_id = response["browserId"]
    print(f"Created new browser: {browser_id}")

except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        print(f"Browser '{BROWSER_NAME}' already exists from a previous run.")
        print("Reusing existing browser. To recreate with a new policy,")
        print("run the Cleanup section first, then re-run this cell.")
        browsers = client.list_browsers(browser_type="CUSTOM")
        browser_id = None
        for b in browsers.get("browserSummaries", []):
            if b.get("name") == BROWSER_NAME:
                browser_id = b["browserId"]
                break
        if not browser_id:
            raise RuntimeError(f"Could not find browser '{BROWSER_NAME}'")
    else:
        raise

print(f"Browser ID: {browser_id}")

브라우저가 **READY** 상태가 될 때까지 기다립니다.

In [ ]:
print("Waiting for browser to become ready...")
while True:
    info = client.get_browser(browser_id)
    status = info["status"]
    if status == "READY":
        print(f"Browser is ready: {browser_id}")
        break
    elif status == "CREATE_FAILED":
        reason = info.get("failureReason", "Unknown")
        print(f"Browser creation failed: {reason}")
        raise SystemExit(1)
    print(f"  Status: {status} — waiting...")
    time.sleep(5)

### 3단계: Playwright로 Chrome policy 적용 확인

브라우저 세션을 시작하고 [Playwright](https://playwright.dev/docs/intro)를 사용해 두 URL로 이동합니다.

1. **`docs.aws.amazon.com`** - Policy에서 허용됨 → 페이지가 정상적으로 load됨
2. **`www.wikipedia.org`** - Policy에서 차단됨 → Chrome이 오류 페이지를 표시함

이를 통해 제한이 Agent prompt나 추론 로직과 관계없이 브라우저 수준에서 적용된다는 것을 확인할 수 있습니다.

> **팁:** 이 셀이 실행되는 동안 AgentCore Console에서 브라우저를 실시간으로 확인할 수 있습니다. **Built-in tools** → **docs_research_browser** → **View live session**으로 이동하세요.

In [ ]:
from bedrock_agentcore.tools import BrowserClient
from playwright.async_api import async_playwright

session_client = BrowserClient(REGION)
session_id = session_client.start(identifier=browser_id, session_timeout_seconds=3600)
print(f"Session started: {session_id}")

# 세션이 준비될 때까지 polling
for i in range(30):
    info = session_client.get_session()
    status = info.get("status")
    print(f"  Status: {status}")
    if status == "READY":
        break
    time.sleep(5)

ws_url, headers = session_client.generate_ws_headers()


async def test_policy_enforcement():
    async with async_playwright() as p:
        browser = await p.chromium.connect_over_cdp(ws_url, headers=headers, timeout=60000)
        print("Connected!")
        context = browser.contexts[0]
        page = context.pages[0] if context.pages else await context.new_page()

        # ── 테스트 1: ALLOWED URL로 이동 ──
        print("\n" + "=" * 60)
        print("TEST 1: Navigate to docs.aws.amazon.com (ALLOWED)")
        print("=" * 60)
        await page.goto(
            "https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html",
            wait_until="domcontentloaded",
            timeout=60000,
        )
        await asyncio.sleep(3)

        title = await page.title()
        print(f"Page title: {title}")

        # 먼저 script/style/noscript 요소를 제거한 뒤 text 추출
        text = await page.evaluate("""() => {
            const scripts = document.querySelectorAll('script, style, noscript');
            scripts.forEach(s => s.remove());
            return document.body.innerText;
        }""")
        print(f"Extracted {len(text)} chars")
        print(f"First 500 chars:\n{text[:500]}")

        # ── 테스트 2: BLOCKED URL로 이동 ──
        print("\n" + "=" * 60)
        print("TEST 2: Navigate to www.wikipedia.org (BLOCKED)")
        print("=" * 60)
        try:
            await page.goto(
                "https://www.wikipedia.org",
                wait_until="domcontentloaded",
                timeout=15000,
            )
            blocked_title = await page.title()
            content = await page.evaluate("() => document.documentElement.outerHTML", None)
            if "blocked" in content.lower() or "ERR_BLOCKED" in content:
                print("Result: CHROME POLICY BLOCKED THIS URL ✅")
            else:
                print(f"Result: Page loaded (unexpected) — title: {blocked_title}")
        except Exception:
            print("Result: CHROME POLICY BLOCKED THIS URL ✅")

        await browser.close()
        return text


docs_text = await test_policy_enforcement()
session_client.stop()
print("\nSession stopped.")

### 4단계: Session recording 검토

2단계에서 session recording을 활성화했으므로 세션을 재생해 policy 적용 결과를 확인할 수 있습니다.

**Amazon Bedrock AgentCore Console에서 녹화를 검토하려면 다음 단계를 따르세요.**

1. [Amazon Bedrock AgentCore Console](https://console.aws.amazon.com/bedrock-agentcore/home#)을 엽니다.
2. Navigation pane에서 **Built-in tools**를 선택합니다.
3. Browser tool(**docs_research_browser**)을 선택합니다.
4. **Browser sessions** 섹션에서 **Terminated** 상태인 완료된 세션을 찾습니다.
5. **View Recording**을 선택합니다.

Replay interface에는 다음 정보가 표시됩니다.
- **Video player** - Timeline scrubber가 포함된 대화형 재생 화면
- **User actions** - 차단된 URL 시도를 포함한 timestamp가 있는 탐색 event
- **Network events** - `docs.aws.amazon.com` traffic만 성공했는지 확인

### (선택 사항) 4b단계: 제한된 브라우저로 Strands Agent 실행

Policy로 제한된 브라우저를 AI Agent framework와 함께 사용할 수도 있습니다. 아래 셀에서는 AgentCore 문서를 조사하는 [Strands](https://strandsagents.com/) Agent를 생성합니다. Agent는 `docs.aws.amazon.com`에서 작업에 성공하고 `wikipedia.org`가 차단되는 것을 확인합니다.

이 sample은 Amazon Bedrock을 통해 Anthropic Claude를 사용합니다. AgentCore는 특정 모델에 종속되지 않으므로 다른 model provider로 대체할 수 있습니다. 모델 구성은 [Model Providers](https://strandsagents.com/latest/user-guide/concepts/model-providers/)를 참고하세요.

> **참고:** `strands-agents-tools`의 `AgentCoreBrowser` tool은 필요할 때 브라우저 세션을 생성합니다. 첫 시도에서 연결 timeout이 발생하면 tool이 재시도합니다. 새로 생성한 브라우저의 첫 세션 생성에는 시간이 더 걸릴 수 있습니다.

In [ ]:
from strands import Agent
from strands_tools.browser import AgentCoreBrowser

SYSTEM_PROMPT = """You are a research assistant that reads AWS documentation
and provides summaries. Navigate to the provided URLs, read the content,
and summarize the key information. Stay focused on the documentation pages
provided. If a page fails to load or is blocked, note that it was blocked
and continue working with the pages that are accessible."""

browser_tool = AgentCoreBrowser(
    region=REGION,
    identifier=browser_id,
)

agent = Agent(
    tools=[browser_tool.browser],
    system_prompt=SYSTEM_PROMPT,
)

prompt = """Research Amazon Bedrock AgentCore Browser by reading the documentation at
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html

Summarize the top 3 capabilities and provide a brief explanation of each.

Also try navigating to https://www.wikipedia.org to find additional context
about AI agents."""

print(f"Prompt: {prompt}\n")
print("Running agent...\n")

response = agent(prompt)

print("\n" + "=" * 60)
print("AGENT RESPONSE:")
print("=" * 60)
print(response.message["content"][0]["text"])

---
## Part 2: Custom Root CA Certificate

Private Certificate Authority로 내부 서비스를 운영하거나 SSL-intercepting corporate proxy를 통해 traffic을 routing하는 조직은 Agent가 해당 비공개 certificate를 신뢰하도록 구성해야 합니다.

이 기능을 내부 infrastructure 없이 확인하기 위해 이 섹션에서는 신뢰할 수 없는 root CA가 서명한 certificate를 의도적으로 사용하는 공개 웹 사이트 [https://untrusted-root.badssl.com](https://untrusted-root.badssl.com)을 사용합니다. 일반적으로 이 사이트에 대한 HTTPS 연결은 SSL certificate 오류로 실패합니다. 올바른 root CA가 없을 때 내부 서비스 연결이 실패하는 것과 같습니다.

### 5단계: AWS Secrets Manager에 root CA certificate 저장

BadSSL의 신뢰할 수 없는 root CA certificate는 공개되어 있습니다(출처: [badssl.com/certs/ca-untrusted-root.crt](https://badssl.com/certs/ca-untrusted-root.crt)). AgentCore가 trusted certificate store로 import할 수 있도록 Secrets Manager에 저장합니다.

실제 조직에서도 같은 방식으로 내부 CA certificate 또는 SSL-intercepting proxy의 root CA certificate를 저장합니다.

In [ ]:
SECRET_NAME = "demo-badssl-untrusted-root-ca"

BADSSL_ROOT_CA = """-----BEGIN CERTIFICATE-----
MIIGfjCCBGagAwIBAgIJAJeg/PrX5Sj9MA0GCSqGSIb3DQEBCwUAMIGBMQswCQYD
VQQGEwJVUzETMBEGA1UECAwKQ2FsaWZvcm5pYTEWMBQGA1UEBwwNU2FuIEZyYW5j
aXNjbzEPMA0GA1UECgwGQmFkU1NMMTQwMgYDVQQDDCtCYWRTU0wgVW50cnVzdGVk
IFJvb3QgQ2VydGlmaWNhdGUgQXV0aG9yaXR5MB4XDTE2MDcwNzA2MzEzNVoXDTM2
MDcwMjA2MzEzNVowgYExCzAJBgNVBAYTAlVTMRMwEQYDVQQIDApDYWxpZm9ybmlh
MRYwFAYDVQQHDA1TYW4gRnJhbmNpc2NvMQ8wDQYDVQQKDAZCYWRTU0wxNDAyBgNV
BAMMK0JhZFNTTCBVbnRydXN0ZWQgUm9vdCBDZXJ0aWZpY2F0ZSBBdXRob3JpdHkw
ggIiMA0GCSqGSIb3DQEBAQUAA4ICDwAwggIKAoICAQDKQtPMhEH073gis/HISWAi
bOEpCtOsatA3JmeVbaWal8O/5ZO5GAn9dFVsGn0CXAHR6eUKYDAFJLa/3AhjBvWa
tnQLoXaYlCvBjodjLEaFi8ckcJHrAYG9qZqioRQ16Yr8wUTkbgZf+er/Z55zi1yn
CnhWth7kekvrwVDGP1rApeLqbhYCSLeZf5W/zsjLlvJni9OrU7U3a9msvz8mcCOX
fJX9e3VbkD/uonIbK2SvmAGMaOj/1k0dASkZtMws0Bk7m1pTQL+qXDM/h3BQZJa5
DwTcATaa/Qnk6YHbj/MaS5nzCSmR0Xmvs/3CulQYiZJ3kypns1KdqlGuwkfiCCgD
yWJy7NE9qdj6xxLdqzne2DCyuPrjFPS0mmYimpykgbPnirEPBF1LW3GJc9yfhVXE
Cc8OY8lWzxazDNNbeSRDpAGbBeGSQXGjAbliFJxwLyGzZ+cG+G8lc+zSvWjQu4Xp
GJ+dOREhQhl+9U8oyPX34gfKo63muSgo539hGylqgQyzj+SX8OgK1FXXb2LS1gxt
VIR5Qc4MmiEG2LKwPwfU8Yi+t5TYjGh8gaFv6NnksoX4hU42gP5KvjYggDpR+NSN
CGQSWHfZASAYDpxjrOo+rk4xnO+sbuuMk7gORsrl+jgRT8F2VqoR9Z3CEdQxcCjR
5FsfTymZCk3GfIbWKkaeLQIDAQABo4H2MIHzMB0GA1UdDgQWBBRvx4NzSbWnY/91
3m1u/u37l6MsADCBtgYDVR0jBIGuMIGrgBRvx4NzSbWnY/913m1u/u37l6MsAKGB
h6SBhDCBgTELMAkGA1UEBhMCVVMxEzARBgNVBAgMCkNhbGlmb3JuaWExFjAUBgNV
BAcMDVNhbiBGcmFuY2lzY28xDzANBgNVBAoMBkJhZFNTTDE0MDIGA1UEAwwrQmFk
U1NMIFVudHJ1c3RlZCBSb290IENlcnRpZmljYXRlIEF1dGhvcml0eYIJAJeg/PrX
5Sj9MAwGA1UdEwQFMAMBAf8wCwYDVR0PBAQDAgEGMA0GCSqGSIb3DQEBCwUAA4IC
AQBQU9U8+jTRT6H9AIFm6y50tXTg/ySxRNmeP1Ey9Zf4jUE6yr3Q8xBv9gTFLiY1
qW2qfkDSmXVdBkl/OU3+xb5QOG5hW7wVolWQyKREV5EvUZXZxoH7LVEMdkCsRJDK
wYEKnEErFls5WPXY3bOglBOQqAIiuLQ0f77a2HXULDdQTn5SueW/vrA4RJEKuWxU
iD9XPnVZ9tPtky2Du7wcL9qhgTddpS/NgAuLO4PXh2TQ0EMCll5reZ5AEr0NSLDF
c/koDv/EZqB7VYhcPzr1bhQgbv1dl9NZU0dWKIMkRE/T7vZ97I3aPZqIapC2ulrf
KrlqjXidwrGFg8xbiGYQHPx3tHPZxoM5WG2voI6G3s1/iD+B4V6lUEvivd3f6tq7
d1V/3q1sL5DNv7TvaKGsq8g5un0TAkqaewJQ5fXLigF/yYu5a24/GUD783MdAPFv
gWz8F81evOyRfpf9CAqIswMF+T6Dwv3aw5L9hSniMrblkg+ai0K22JfoBcGOzMtB
Ke/Ps2Za56dTRoY/a4r62hrcGxufXd0mTdPaJLw3sJeHYjLxVAYWQq4QKJQWDgTS
dAEWyN2WXaBFPx5c8KIW95Eu8ShWE00VVC3oA4emoZ2nrzBXLrUScifY6VaYYkkR
2O2tSqU8Ri3XRdgpNPDWp8ZL49KhYGYo3R/k98gnMHiY5g==
-----END CERTIFICATE-----"""

try:
    sm_client.create_secret(
        Name=SECRET_NAME,
        SecretString=BADSSL_ROOT_CA,
        Description="BadSSL untrusted root CA for demo purposes",
    )
    print(f"Created secret: {SECRET_NAME}")
except sm_client.exceptions.ResourceExistsException:
    print(f"Secret already exists: {SECRET_NAME}")

secret_arn = sm_client.describe_secret(SecretId=SECRET_NAME)["ARN"]
print(f"Secret ARN: {secret_arn}")

### 6단계: Root CA가 없는 Code Interpreter - SSL 오류 예상

기본 Code Interpreter 세션을 생성하고 `https://untrusted-root.badssl.com`에 연결을 시도합니다. Root CA를 신뢰하지 않으므로 연결이 실패합니다.

In [ ]:
from bedrock_agentcore.tools import CodeInterpreter

TEST_CODE = 'import urllib.request\ntry:\n    response = urllib.request.urlopen("https://untrusted-root.badssl.com")\n    print(f"Status: {response.status}")\nexcept Exception as e:\n    print(f"Error: {type(e).__name__}")\n    print(f"The connection failed because the root CA is not trusted.")'

ci_client = CodeInterpreter(REGION)
ci_client.start()

print(f"Session started: {ci_client.session_id}")
print("Connecting to https://untrusted-root.badssl.com ...\n")

result = ci_client.invoke(
    "executeCode",
    {
        "code": TEST_CODE,
        "language": "python",
    },
)

for event in result.get("stream", []):
    if "result" in event:
        content = event["result"]
        is_error = content.get("isError", False)
        structured = content.get("structuredContent", {})
        stderr = structured.get("stderr", "")
        stdout = structured.get("stdout", "")

        if (
            is_error
            or "SSLCertVerificationError" in stderr
            or "SSLCertVerificationError" in stdout
            or "Error:" in stdout
        ):
            print("Result: SSL CERTIFICATE ERROR (expected)")
            print("  The connection failed because the root CA is not trusted.")
        else:
            print(f"  stdout: {stdout[:200]}")

ci_client.stop()
print("\nSession stopped.")

### 7단계: Root CA가 있는 Code Interpreter - HTTP 200 예상

`certificates` parameter를 사용해 BadSSL root CA certificate를 신뢰하는 사용자 지정 Code Interpreter를 생성합니다. Browser의 `create_browser()` 호출에서 사용한 것과 동일한 `Certificate.from_secret_arn(...)` pattern을 사용합니다.


In [ ]:
from bedrock_agentcore.tools import Certificate

ci_client_with_ca = CodeInterpreter(REGION)

response = ci_client_with_ca.create_code_interpreter(
    name="demo_rootca_interpreter",
    execution_role_arn=EXECUTION_ROLE_ARN,
    network_configuration={"networkMode": "PUBLIC"},
    certificates=[Certificate.from_secret_arn(secret_arn)],
    description="Code interpreter trusting BadSSL untrusted root CA",
)

interpreter_id = response["codeInterpreterId"]
print(f"Created interpreter: {interpreter_id}")

# 준비될 때까지 대기
print("Waiting for interpreter to become ready...")
while True:
    info = ci_client_with_ca.get_code_interpreter(interpreter_id)
    if info["status"] == "READY":
        print("Interpreter is ready.")
        break
    elif info["status"] == "CREATE_FAILED":
        print(f"Failed: {info.get('failureReason', 'Unknown')}")
        raise SystemExit(1)
    time.sleep(3)

사용자 지정 interpreter로 세션을 시작하고 동일한 코드를 실행합니다.

In [ ]:
SUCCESS_CODE = 'import urllib.request\nresponse = urllib.request.urlopen("https://untrusted-root.badssl.com")\nprint(f"Status: {response.status}")\nprint(response.read().decode("utf-8")[:200])'

ci_client_with_ca.start(identifier=interpreter_id)
print(f"Session started: {ci_client_with_ca.session_id}")
print("Connecting to https://untrusted-root.badssl.com ...\n")

result = ci_client_with_ca.invoke(
    "executeCode",
    {
        "code": SUCCESS_CODE,
        "language": "python",
    },
)

for event in result.get("stream", []):
    if "result" in event:
        content = event["result"]
        structured = content.get("structuredContent", {})
        stdout = structured.get("stdout", "")
        exit_code = structured.get("exitCode", -1)

        if exit_code == 0 and "200" in stdout:
            print("Result: SUCCESS — HTTP 200")
            print("  The connection succeeded because the root CA is now trusted.")
            print(f"  Output: {stdout[:200]}")
        else:
            print(f"  Unexpected result (exit code {exit_code}): {stdout[:300]}")

ci_client_with_ca.stop()
print("\nSession stopped.")

### 조직 환경에 적용

`badssl.com` demo는 다음 두 가지 실제 시나리오를 재현합니다.

| 시나리오 | Secrets Manager에 저장할 항목 | 구성 |
|----------|-------------------------------------|---------------|
| 내부 기업 서비스 | 조직의 root CA certificate(HR portal, Jira, Artifactory) | Browser 또는 Code Interpreter를 생성할 때 `certificates`에서 secret ARN 참조 |
| SSL-intercepting corporate proxy | Proxy의 root CA certificate(Zscaler, Palo Alto Networks) | `certificates`에서 secret ARN을 참조하고 proxy 설정 구성 |

단일 `create_browser` 호출에서 root CA certificate와 Chrome policy를 함께 사용할 수 있습니다. 통합 예제는 함께 제공되는 블로그 게시물을 참고하세요.

---
## 정리

비용이 발생하지 않도록 이 Notebook에서 생성한 모든 리소스를 삭제합니다.

In [ ]:
# 활성 세션을 모두 중지한 다음 사용자 지정 브라우저 삭제
try:
    print(f"Stopping active sessions for browser: {browser_id}")
    sessions = client.list_sessions(browser_id=browser_id, status="READY")
    for s in sessions.get("items", []):
        sid = s["sessionId"]
        print(f"  Stopping session: {sid}")
        try:
            client.data_plane_client.stop_browser_session(browserIdentifier=browser_id, sessionId=sid)
        except Exception as e:
            print(f"    Error: {e}")

    if sessions.get("items"):
        print("Waiting for sessions to terminate...")
        time.sleep(15)

    print(f"Deleting browser: {browser_id}")
    client.delete_browser(browser_id)
    print("  Deleted.")
except Exception as e:
    print(f"  Could not delete browser: {e}")

In [ ]:
# Part 2에서 생성한 사용자 지정 Code Interpreter 삭제
try:
    print(f"Deleting interpreter: {interpreter_id}")
    ci_client_with_ca.delete_code_interpreter(interpreter_id)
    print("  Deleted.")
except NameError:
    print("  No interpreter to delete (Part 2 was not run).")
except Exception as e:
    print(f"  Could not delete interpreter: {e}")

In [ ]:
# Secrets Manager secret 삭제
try:
    print(f"Deleting secret: {SECRET_NAME}")
    sm_client.delete_secret(
        SecretId=SECRET_NAME,
        ForceDeleteWithoutRecovery=True,
    )
    print("  Deleted.")
except NameError:
    print("  No secret to delete (Part 2 was not run).")
except Exception as e:
    print(f"  Could not delete secret: {e}")

In [ ]:
# S3 policy file 삭제
print(f"Deleting policy: s3://{BUCKET_NAME}/{POLICY_KEY}")
try:
    s3_client.delete_object(Bucket=BUCKET_NAME, Key=POLICY_KEY)
    print("  Deleted.")
except Exception as e:
    print(f"  Could not delete policy: {e}")

In [ ]:
# Policy를 먼저 분리한 다음 IAM role 삭제
try:
    print(f"Deleting IAM role: {AC_ROLE_NAME}")
    # Inline policy 삭제
    try:
        iam_client.delete_role_policy(RoleName=AC_ROLE_NAME, PolicyName="ac_browser_s3_policy")
    except Exception:
        pass
    # Role 삭제
    iam_client.delete_role(RoleName=AC_ROLE_NAME)
    print("  Deleted.")
except Exception as e:
    print(f"  Could not delete role: {e}")

print("\nCleanup complete.")